# EasyRAG Embedding Inputs And Provider Behavior

## What you will do

- compare plain-text and multimodal embedding inputs
- inspect how EasyRAG builds provider-facing payloads from chunk metadata
- show how provider behavior changes what the vector backend can index

## Why this matters

Embedding quality is not only about the model name. It is also about what input object the model actually receives.


## Source code anchors

- `easyrag.rag.indexing.pipeline._build_embedding_input`
- `easyrag.rag.providers.default_embedding_func`
- `easyrag.config.models.get_embedding_model_name`
- `easyrag.config.models.get_embedding_base_url`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.config import get_embedding_base_url, get_embedding_model_name
from easyrag.rag.indexing.pipeline import build_insert_payloads
from easyrag.support.optional_deps import Document


## Deterministic path

We first inspect the input contract itself. One document is plain text. One document carries `image_paths`, which makes the embedding input multimodal even before any provider is called.


In [ ]:
input_tmp = tempfile.TemporaryDirectory()
image_path = Path(input_tmp.name) / "diagram.png"
image_path.write_bytes(b"PNGDATA")

plain_document = Document(
    page_content="# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.",
    metadata={"doc_id": "doc::plain", "path": "docs/retrieval.md", "relative_path": "docs/retrieval.md", "title": "retrieval", "source_type": "doc"},
)
multimodal_document = Document(
    page_content="Scanned PDF page 1 from manual.",
    metadata={
        "doc_id": "doc::pdf::page::1",
        "path": "docs/manual.pdf",
        "relative_path": "docs/manual.pdf",
        "title": "manual",
        "source_type": "pdf",
        "page_number": 1,
        "image_paths": [str(image_path)],
        "has_visual_content": True,
    },
)
input_rag = EasyRAG(working_dir=input_tmp.name, workspace="embedding-inputs-demo", embedding_func=_stub_embedding, query_model_func=_stub_query_model)
payloads = build_insert_payloads(input_rag, [plain_document, multimodal_document])
_print_json(payloads["vector_chunks"])


## Output inspection

The plain-text chunk produces a string embedding input. The PDF-like chunk produces a dictionary with text plus image paths. That difference is what later lets a multimodal provider preserve page images instead of throwing them away.


In [ ]:
provider_summary = {
    "embedding_model_name": get_embedding_model_name(),
    "embedding_base_url": get_embedding_base_url(),
    "plain_input_type": type(payloads["vector_chunks"][0]["embedding_input"]).__name__,
    "multimodal_input_type": type(payloads["vector_chunks"][1]["embedding_input"]).__name__,
}
_print_json(provider_summary)


## Provider-backed path

If the environment is configured, the next cell sends the same inputs to the repo's default embedding function. The notebook only inspects the returned shape because provider-specific values vary.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_rag = EasyRAG(workspace="provider-embedding-inputs-demo")
    try:
        provider_vectors = provider_rag.embedding_func([item["embedding_input"] for item in payloads["vector_chunks"]])
        _print_json({"vector_lengths": [len(vector) for vector in provider_vectors]})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")


## What changed and why

The key change is not the vector values themselves. It is the input object contract. Once `image_paths` survive into the embedding input, a multimodal provider can use visual context that a plain text-only provider would never see.


In [ ]:
input_tmp.cleanup()
print("Cleaned up the temporary embedding-input workspace.")


## Next steps

- Continue with [03_06_vector_index_basics.ipynb](03_06_vector_index_basics.ipynb) to see how those embedding outputs become searchable records.
- Read [03-indexing-overview.md](../../docs/03-indexing-overview.md) for the stage-level indexing map.
- Revisit [02_03_pdf_and_multimodal_loading.ipynb](../02_data_loading/02_03_pdf_and_multimodal_loading.ipynb) if you want the loading-side view of these image paths.
